# Starter notebook

Copy this file to start a new model. It shows the four things every notebook here does:

1. Import `bootstrap`, which makes the rest of the project importable.
2. Load a ready-made table from `research/datasets.py` (cached to disk after the first run).
3. Reach the app's services directly, for anything the ready-made tables don't cover.
4. Cache your own expensive table with `snapshot`, so a kernel restart is cheap.

Setup, once: `pip install -r requirements-research.txt`, then run `jupyter lab` from the repo root.

## 1. Bootstrap

`bootstrap` lives next to this notebook, so it always imports. It puts the repo folder on Python's search path, which is what makes `research`, `services`, `repositories` and the rest importable.

The `autoreload` lines mean that editing a project `.py` file takes effect in this kernel without restarting it.

In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from bootstrap import get_context, dfs_player_weeks, snapshot, list_snapshots

# Show every column when printing a frame -- these tables are very wide.
pd.set_option("display.max_columns", 200)

## 2. Load a ready-made table

The first call takes a couple of minutes: it downloads several seasons of play-by-play from nflverse and joins five sources together. It then writes the result to `data/research/dfs_player_weeks_FanDuel.parquet`, and every later call — including after a kernel restart — reads that file in under a second.

Pass `refresh=True` after a week of games to rebuild it.

In [ ]:
# One row per player per week, 2023-2025. Always has canonical_id, name,
# position, team, opponent, season, week; then fantasy points, usage shares,
# snap counts, red-zone touches, and tracking/charting columns where the
# source had something to say (blank where it did not).
weeks = dfs_player_weeks()

print(weeks.shape)
weeks.head()

In [ ]:
# What's actually in here, and how much of each column is filled in.
coverage = weeks.notna().mean().sort_values(ascending=False)
coverage.to_frame("filled_in").head(40)

## 3. A baseline model

A worked example, not a good model: predict a receiver's fantasy points this week from last week's usage.

The one thing worth copying is the **split**. Football data is a time series, so train on earlier seasons and test on a later one. A random train/test split would let the model peek at the future and score far better than it deserves.

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error

# What the model predicts, and what it predicts from. The target is in the
# feature list too, so last week's score becomes a feature (and a baseline).
TARGET = "total_fantasy_points"
FEATURES = ["targets", "receptions", "receiving_yards", "target_share",
            "snap_share", TARGET]

# Wide receivers only, sorted so "the row before" really is the previous week.
wr = weeks[weeks["position"] == "WR"].sort_values(["canonical_id", "season", "week"]).copy()

# shift(1) within each player: last week's numbers on this week's row. This is
# what keeps the model honest -- it only ever sees the past.
for column in FEATURES:
    wr[f"prev_{column}"] = wr.groupby("canonical_id")[column].shift(1)

# Drop rows missing any input: week 1, and anyone whose snap counts didn't
# resolve. dropna on the target too -- there is nothing to learn from a blank.
model_frame = wr.dropna(subset=[f"prev_{c}" for c in FEATURES] + [TARGET])

train = model_frame[model_frame["season"] < 2025]
test = model_frame[model_frame["season"] == 2025]

print(f"train rows: {len(train):,}   test rows: {len(test):,}")

In [ ]:
prev_features = [f"prev_{c}" for c in FEATURES]

model = GradientBoostingRegressor(random_state=0)
model.fit(train[prev_features], train[TARGET])

predicted = model.predict(test[prev_features])

# Mean absolute error: on average, how many fantasy points the guess is off by.
# Always compare against the dumbest possible baseline -- here, "he'll score
# what he scored last week" -- because a model that can't beat that is noise.
print(f"model MAE:    {mean_absolute_error(test[TARGET], predicted):.2f}")
print(f"baseline MAE: {mean_absolute_error(test[TARGET], test[f'prev_{TARGET}']):.2f}")

# Which inputs the model actually leaned on.
pd.Series(model.feature_importances_, index=prev_features).sort_values(ascending=False)

## 4. Reaching the app directly

`get_context()` hands back the same object the Streamlit pages use, so anything a page can compute, this notebook can too. It is built once per kernel — the first call is the slow one.

See [app_context.py](../app_context.py) for every service on it.

In [ ]:
ctx = get_context()

# Anything on the context is fair game, e.g. the raw play-by-play behind the
# DFS tables: one row per play, ~140 columns, for the DFS seasons.
plays = ctx.dfs_read_repo.pbp()
plays.shape

## 5. Caching your own table

Anything slow you build yourself belongs in a `snapshot`. Give it a name that includes any setting that changes the contents, and pass a function that builds it (note the `lambda:` — without it the slow work runs before caching can help).

Once a table proves useful, move it into [research/datasets.py](../research/datasets.py) so other notebooks get it too.

In [ ]:
def build_red_zone_rates():
    """Season-level red-zone usage per player."""
    return (weeks.groupby(["canonical_id", "name", "position", "season"], as_index=False)
                 .agg(red_zone_touches=("red_zone_touches", "sum"),
                      games=("week", "count")))


red_zone = snapshot("red_zone_rates", build_red_zone_rates)
red_zone.sort_values("red_zone_touches", ascending=False).head(10)

In [ ]:
# Everything currently cached on disk, newest first. Delete any of it any time --
# clear_snapshots() wipes the lot, and each table rebuilds on next use.
list_snapshots()